In [1]:
import os
os.getcwd()


'e:\\infosys\\infosys-carlease-contract-ai-group2\\notebook'

In [2]:
import pandas as pd
df = pd.read_csv("E:\infosys\infosys-carlease-contract-ai-group2\data\contract_evaluation_dataset_sri.csv")

df.head()

<>:2: SyntaxWarning: invalid escape sequence '\i'
<>:2: SyntaxWarning: invalid escape sequence '\i'
C:\Users\sriha\AppData\Local\Temp\ipykernel_13372\1565292221.py:2: SyntaxWarning: invalid escape sequence '\i'
  df = pd.read_csv("E:\infosys\infosys-carlease-contract-ai-group2\data\contract_evaluation_dataset_sri.csv")


,contract_id,raw_text,expected_apr,expected_term,expected_payment,expected_penalty
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25
4,5,This contract between National Motors and John...,5.97,36,1180,NaN


In [3]:
df.columns

Index(['contract_id', 'raw_text', 'expected_apr', 'expected_term',
       'expected_payment', 'expected_penalty'],
      dtype='object')

In [4]:
expected_output = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}


In [5]:
PROMPT = """
You are an information extraction system.

Extract ONLY the following fields from the contract text:
- APR
- Term (in months)
- Monthly payment
- Penalty clause

Rules:
- Return output strictly in JSON format
- If a value is not mentioned, return null
- Do NOT infer or assume values

Return JSON with keys:
apr, term_months, monthly_payment, penalty_clause
"""

In [6]:
import re

def extract_sla(contract_text):
    result = {
        "apr": None,
        "term_months": None,
        "monthly_payment": None,
        "penalty_clause": None
    }

    apr = re.search(r'APR\s*([\d.]+)%', contract_text)
    if apr:
        result["apr"] = float(apr.group(1))

    term = re.search(r'(\d+)\s*months', contract_text)
    if term:
        result["term_months"] = int(term.group(1))

    payment = re.search(r'\$\s*(\d+)', contract_text)
    if payment:
        result["monthly_payment"] = int(payment.group(1))

    penalty = re.search(r'(late fee.*?\$\d+|early termination.*?\$\d+)', contract_text, re.IGNORECASE)
    if penalty:
        result["penalty_clause"] = penalty.group(0)

    return result

In [8]:
sample_df = df[[
    "raw_text",
    "expected_apr",
    "expected_term",
    "expected_payment",
    "expected_penalty"
]].sample(5, random_state=42).reset_index(drop=True)

In [9]:
results = []

for text in sample_df["raw_text"]:
    results.append(extract_sla(text))

results_df = pd.DataFrame(results)
results_df


,apr,term_months,monthly_payment,penalty_clause
0,3.76,48,664,None
1,9.11,24,896,Late fee $25
2,10.49,24,959,None
3,6.40,48,648,Early termination fee $300
4,12.77,48,610,None


In [10]:
evaluation_df = sample_df.copy()

evaluation_df["apr_match"] = (results_df["apr"] == evaluation_df["expected_apr"]).astype(int)
evaluation_df["term_match"] = (results_df["term_months"] == evaluation_df["expected_term"]).astype(int)
evaluation_df["payment_match"] = (results_df["monthly_payment"] == evaluation_df["expected_payment"]).astype(int)
evaluation_df["penalty_match"] = (results_df["penalty_clause"] == evaluation_df["expected_penalty"]).astype(int)

evaluation_df[[
    "apr_match",
    "term_match",
    "payment_match",
    "penalty_match"
]]

,apr_match,term_match,payment_match,penalty_match
0,1,1,1,0
1,1,1,1,1
2,1,1,1,0
3,1,1,1,1
4,1,1,1,0
